# AEMCF: Adaptive Energy Management and Control Framework for Long-Endurance UAVs

Simulation of a hybrid battery/fuel-cell UAV energy management system,
combining a wind-compensating guidance law with a convex-QP power-allocation
model predictive controller (MPC).

**Contents:**
1. Install dependencies
2. Simulation module (vehicle dynamics, wind process, energy model, guidance law, QP controller)
3. Monte Carlo experiment driver
4. Run an experiment
5. Summary statistics and paired significance tests
6. Derived metrics (on-station loiter time, energy-budget utilisation)
7. Plots
8. Notes on extending the experiment

**Runtime:** each seed solves a QP once per simulated second for both
MPC-based controllers (~7,000–7,500 solves per seed over a ~2-hour
mission). On a Colab CPU runtime this is roughly 30–40 seconds per seed,
so `N_RUNS = 50` takes on the order of 25–35 minutes. Reduce `N_RUNS`
for a quick check.

In [ ]:
!pip install -q cvxpy


## 1. Simulation module

In [ ]:
%%writefile uav_energy_sim.py
"""
AEMCF simulation: hybrid battery/fuel-cell UAV energy management.

Two layers:
1. Guidance law - nonlinear, closed-form, wind-compensating heading + speed
2. Power-allocation QP - convex, solved every step via OSQP/cvxpy

Also includes two baselines (fixed 50/50 split, static offline split)
and a wind-blind ablation of AEMCF.

Units: SI (m, s, kg, W, J), except energy in Wh and SOC as a fraction.
"""

import numpy as np
import cvxpy as cp
import time

# --- vehicle / battery / fuel cell params (illustrative, not a real airframe) ---
MASS = 5.5                 # kg
WING_AREA = 0.8             # m^2
RHO = 1.225                 # air density, kg/m^3
CD = 0.045                  # drag coefficient
ETA_PROP = 0.75              # propulsion efficiency
G = 9.81

BATT_VNOM = 22.2             # V
BATT_CAP_AH = 6.0             # Ah
BATT_CAP_WH = BATT_VNOM * BATT_CAP_AH   # 133.2 Wh
ETA_DIS = 0.95               # battery discharge efficiency

FC_RATED = 120.0             # W
FC_MIN = 0.0
FC_MAX = FC_RATED
FC_RAMP = 15.0                # W/s slew limit
FC_EFF_TARGET = 0.70          # most efficient load point
FC_FUEL_WH = 250.0            # onboard fuel budget, Wh
FC_ETA_PEAK = 0.55
FC_ETA_CURVATURE = 0.22

def fc_efficiency(p_fc):
    """PEM fuel cell efficiency curve, peaks near FC_EFF_TARGET."""
    load_frac = np.clip(p_fc, 1e-3, FC_RATED) / FC_RATED
    eta = FC_ETA_PEAK - FC_ETA_CURVATURE * (load_frac - FC_EFF_TARGET) ** 2
    return float(np.clip(eta, 0.15, FC_ETA_PEAK))

P_PAYLOAD = 10.0             # W
P_COMM = 5.0                  # W

V_CRUISE = 15.0               # m/s
V_MIN = 10.0
V_MAX = 20.0
GAMMA_MAX = np.deg2rad(8.0)     # max climb/descent angle

SOC_MIN = 0.20
SOC_MAX = 0.95
SOC_INIT = 0.95

DT = 1.0                      # sim step, s
HORIZON_N = 15                 # MPC horizon, steps

WAYPOINT_CAPTURE_RADIUS = 150.0  # m


def default_mission():
    # 8 waypoints, ~20x20 km area, altitude 150-300 m
    pts = np.array([
        [0,     0,    150],
        [3000,  4200, 220],
        [7500,  2500, 180],
        [11000, 6000, 300],
        [14500, 3200, 200],
        [17000, 8000, 260],
        [19000, 12000, 180],
        [20000, 18000, 150],
    ], dtype=float)
    return pts

LOITER_RADIUS = 300.0  # holding pattern radius after last waypoint, m


class WindProcess:
    """Simplified Gauss-Markov (OU) gust model, not a full Dryden spectrum."""

    def __init__(self, rng, mean_speed=4.0, max_speed=7.5, tau=55.0, dt=DT, sigma=1.6):
        self.rng = rng
        self.tau = tau
        self.dt = dt
        self.max_speed = max_speed
        theta0 = rng.uniform(0, 2 * np.pi)
        speed0 = np.clip(rng.normal(mean_speed, 1.5), 0, max_speed)
        self.w = np.array([speed0 * np.cos(theta0), speed0 * np.sin(theta0)])
        self.sigma = sigma

    def step(self):
        a = np.exp(-self.dt / self.tau)
        noise = self.rng.normal(0, self.sigma * np.sqrt(1 - a ** 2), size=2)
        self.w = a * self.w + noise
        speed = np.linalg.norm(self.w)
        if speed > self.max_speed:
            self.w = self.w / speed * self.max_speed
        return self.w.copy()


def drag_force(v_air):
    return 0.5 * RHO * v_air ** 2 * CD * WING_AREA

def propulsion_power(v_air, climb_rate):
    D = drag_force(v_air)
    p_level = D * v_air / ETA_PROP
    p_climb = max(0.0, MASS * G * climb_rate) / ETA_PROP  # no regen on descent
    return p_level + p_climb

def total_power(v_air, climb_rate):
    return propulsion_power(v_air, climb_rate) + P_PAYLOAD + P_COMM


def guidance_step(pos, wp, wind_est, soc, wind_aware=True, speed_schedule=True):
    """Nonlinear guidance law, solved in closed form (not part of the QP)."""
    to_wp = wp - pos
    dist_xy = np.linalg.norm(to_wp[:2])
    track_dir = to_wp[:2] / (dist_xy + 1e-6)

    w = wind_est if wind_aware else np.zeros(2)

    v_air_cmd = V_CRUISE
    if speed_schedule:
        # slow into headwind when SOC is low, speed up with tailwind when SOC is healthy
        headwind_component = -np.dot(w, track_dir)
        soc_factor = np.clip((soc - SOC_MIN) / (SOC_MAX - SOC_MIN), 0, 1)
        adj = -0.35 * headwind_component / 7.5 * (1.3 - soc_factor)
        v_air_cmd = np.clip(V_CRUISE * (1 + adj * 0.2), V_MIN, V_MAX)

    # crab angle so ground track matches track_dir given wind
    wperp = w[0] * (-track_dir[1]) + w[1] * track_dir[0]
    ratio = np.clip(wperp / max(v_air_cmd, 1e-3), -0.98, 0.98)
    crab = np.arcsin(ratio)
    psi = np.arctan2(track_dir[1], track_dir[0]) + crab

    heading_vec = np.array([np.cos(psi), np.sin(psi)])
    ground_vec_xy = v_air_cmd * heading_vec + w

    dz = to_wp[2]
    horiz_time = max(dist_xy / max(np.linalg.norm(ground_vec_xy), 1e-3), 1.0)
    desired_climb = np.clip(dz / horiz_time, -v_air_cmd * np.sin(GAMMA_MAX),
                             v_air_cmd * np.sin(GAMMA_MAX))

    reached = dist_xy < WAYPOINT_CAPTURE_RADIUS
    return psi, v_air_cmd, ground_vec_xy, desired_climb, reached


def loiter_step(pos, center, wind_est, soc, wind_aware=True, speed_schedule=True):
    """Circular holding pattern around `center`, radius LOITER_RADIUS."""
    w = wind_est if wind_aware else np.zeros(2)
    radial = pos[:2] - center[:2]
    r = np.linalg.norm(radial)
    radial_dir = radial / (r + 1e-6)
    tangent_dir = np.array([-radial_dir[1], radial_dir[0]])

    v_air_cmd = V_CRUISE
    if speed_schedule:
        headwind_component = -np.dot(w, tangent_dir)
        soc_factor = np.clip((soc - SOC_MIN) / (SOC_MAX - SOC_MIN), 0, 1)
        adj = -0.35 * headwind_component / 7.5 * (1.3 - soc_factor)
        v_air_cmd = np.clip(V_CRUISE * (1 + adj * 0.2), V_MIN, V_MAX)

    radial_err = (LOITER_RADIUS - r)
    k_radial = 0.15
    desired_dir = tangent_dir + k_radial * (radial_err / LOITER_RADIUS) * (-radial_dir)
    desired_dir = desired_dir / (np.linalg.norm(desired_dir) + 1e-9)

    wperp = w[0] * (-desired_dir[1]) + w[1] * desired_dir[0]
    ratio = np.clip(wperp / max(v_air_cmd, 1e-3), -0.98, 0.98)
    crab = np.arcsin(ratio)
    psi = np.arctan2(desired_dir[1], desired_dir[0]) + crab
    heading_vec = np.array([np.cos(psi), np.sin(psi)])
    ground_vec_xy = v_air_cmd * heading_vec + w
    return ground_vec_xy, 0.0, v_air_cmd


BATT_USABLE_WH = BATT_CAP_WH * (SOC_MAX - SOC_MIN)
FC_ETA_NOMINAL = 0.50  # nominal efficiency used inside the QP forecast only;
                        # actual fuel accounting uses the real fc_efficiency() curve

class PowerMPC:
    """Convex QP for battery/fuel-cell power split, solved every step (OSQP/cvxpy).

    Balances fractional depletion of battery vs. fuel budget (ECMS-style),
    so one source isn't drained while the other still has capacity.
    """

    def __init__(self, N=HORIZON_N, dt=DT,
                 w_balance=15.0, w_fc_band=0.0, w_du=0.02):
        self.N = N
        self.dt = dt
        self.P_batt = cp.Variable(N)
        self.P_demand = cp.Parameter(N)
        self.soc0 = cp.Parameter()
        self.fuel0 = cp.Parameter()
        self.fc_prev = cp.Parameter()

        k_soc = ETA_DIS * dt / (BATT_CAP_WH * 3600.0)
        k_fuel = dt / (3600.0 * FC_ETA_NOMINAL)

        fc = self.P_demand - self.P_batt
        soc_vec = self.soc0 - k_soc * cp.cumsum(self.P_batt)
        fuel_vec = self.fuel0 - k_fuel * cp.cumsum(fc)
        batt_stress = (SOC_INIT - soc_vec) / (SOC_INIT - SOC_MIN)
        fuel_stress = (FC_FUEL_WH - fuel_vec) / FC_FUEL_WH

        cost = w_balance * cp.sum_squares(batt_stress - fuel_stress)
        if w_fc_band > 0:
            cost += w_fc_band * cp.sum_squares(fc / FC_RATED - FC_EFF_TARGET)
        if w_du > 0 and N > 1:
            cost += w_du * cp.sum_squares(cp.diff(self.P_batt))

        fc_full = cp.hstack([cp.reshape(self.fc_prev, (1,), order='C'), fc])
        constraints = [
            fc >= FC_MIN, fc <= FC_MAX,
            cp.diff(fc_full) <= FC_RAMP * dt,
            cp.diff(fc_full) >= -FC_RAMP * dt,
            self.P_batt >= 0, self.P_batt <= self.P_demand,
        ]

        self.problem = cp.Problem(cp.Minimize(cost), constraints)

    def solve(self, soc0, fuel0, fc_prev, p_demand_forecast):
        self.soc0.value = soc0
        self.fuel0.value = fuel0
        self.fc_prev.value = fc_prev
        self.P_demand.value = p_demand_forecast
        t0 = time.perf_counter()
        try:
            self.problem.solve(solver=cp.OSQP, warm_start=True, verbose=False,
                                max_iter=4000)
            p_batt0 = self.P_batt.value[0]
        except Exception:
            p_batt0 = 0.5 * p_demand_forecast[0]
        elapsed = time.perf_counter() - t0
        if p_batt0 is None or np.isnan(p_batt0):
            p_batt0 = 0.5 * p_demand_forecast[0]
        p_batt0 = float(np.clip(p_batt0, 0, p_demand_forecast[0]))
        return p_batt0, elapsed


def run_mission(method, seed, max_time=12000.0, mission=None, mpc=None, wind_kwargs=None):
    """method: 'baseline_fixed', 'baseline_offline', 'aemcf', or 'aemcf_nowind'."""
    rng = np.random.default_rng(seed)
    wind = WindProcess(rng, **(wind_kwargs or {}))
    wp_list = mission if mission is not None else default_mission()

    pos = wp_list[0].copy()
    soc = SOC_INIT
    fc_prev = 0.5 * (propulsion_power(V_CRUISE, 0) + P_PAYLOAD + P_COMM)
    wp_idx = 1
    t = 0.0
    energy_J = 0.0
    fuel_used_Wh = 0.0
    comp_times = []
    soc_trace = []
    fuel_trace = []
    wind_speeds = []

    wind_aware = method in ("aemcf",)
    speed_schedule = method in ("aemcf",)
    use_mpc = method in ("aemcf", "aemcf_nowind")

    # offline baseline: static split computed once from nominal cruise demand
    if method == "baseline_offline":
        p_tot0 = total_power(V_CRUISE, 0)
        denom = ETA_DIS * BATT_USABLE_WH + FC_ETA_NOMINAL * FC_FUEL_WH
        fixed_batt_ratio = (ETA_DIS * BATT_USABLE_WH / denom)
    elif method == "baseline_fixed":
        fixed_batt_ratio = 0.5
    else:
        fixed_batt_ratio = None

    mission_complete_time = None

    while t < max_time and soc > SOC_MIN:
        w_true = wind.step()
        wind_speeds.append(np.linalg.norm(w_true))
        w_est = w_true + rng.normal(0, 0.3, size=2)  # sensor noise
        w_est_used = w_est if (wind_aware) else np.zeros(2)

        in_search_phase = wp_idx < len(wp_list)

        if in_search_phase:
            target = wp_list[wp_idx]
            psi, v_air_cmd, ground_vec_xy, climb_rate, reached = guidance_step(
                pos, target, w_est_used, soc,
                wind_aware=wind_aware, speed_schedule=speed_schedule
            )
            if method in ("baseline_fixed", "baseline_offline"):
                # baselines fly straight, no wind compensation
                to_wp = target - pos
                dist_xy = np.linalg.norm(to_wp[:2])
                track_dir = to_wp[:2] / (dist_xy + 1e-6)
                v_air_cmd = V_CRUISE
                heading_vec = track_dir
                ground_vec_xy = v_air_cmd * heading_vec + w_true
                horiz_time = max(dist_xy / max(np.linalg.norm(ground_vec_xy), 1e-3), 1.0)
                climb_rate = np.clip(to_wp[2] / horiz_time,
                                      -v_air_cmd * np.sin(GAMMA_MAX),
                                      v_air_cmd * np.sin(GAMMA_MAX))
                reached = dist_xy < WAYPOINT_CAPTURE_RADIUS
        else:
            # waypoints done, loiter until battery or fuel runs out
            if mission_complete_time is None:
                mission_complete_time = t
            center = wp_list[-1]
            w_used_loiter = w_true if method in ("baseline_fixed", "baseline_offline") else w_est_used
            ground_vec_xy, climb_rate, v_air_cmd = loiter_step(
                pos, center, w_used_loiter, soc,
                wind_aware=wind_aware if method in ("aemcf", "aemcf_nowind") else False,
                speed_schedule=speed_schedule if method in ("aemcf",) else False,
            )
            reached = False

        p_total = total_power(v_air_cmd, climb_rate)

        if use_mpc:
            forecast = np.full(HORIZON_N, p_total)
            fuel_remaining_now = FC_FUEL_WH - fuel_used_Wh
            p_batt, ct = mpc.solve(soc, fuel_remaining_now, fc_prev, forecast)
            comp_times.append(ct)
        else:
            p_batt = fixed_batt_ratio * p_total
            p_batt = min(p_batt, p_total)

        p_fc = p_total - p_batt
        p_fc = float(np.clip(p_fc, 0, FC_MAX))
        p_batt = p_total - p_fc

        soc -= (p_batt * DT * ETA_DIS) / (BATT_CAP_WH * 3600.0)
        fc_prev = p_fc
        energy_J += p_total * DT

        if p_fc > 1e-6:
            eta_fc = fc_efficiency(p_fc)
            fuel_used_Wh += (p_fc * DT / 3600.0) / eta_fc
        fuel_remaining = FC_FUEL_WH - fuel_used_Wh
        fuel_trace.append(fuel_remaining)

        pos = pos + np.array([ground_vec_xy[0], ground_vec_xy[1], climb_rate]) * DT
        t += DT
        soc_trace.append(soc)

        if in_search_phase and reached:
            wp_idx += 1
        if soc <= SOC_MIN or fuel_remaining <= 0:
            break

    completed = (wp_idx >= len(wp_list))
    energy_Wh = energy_J / 3600.0
    result = {
        "method": method,
        "seed": seed,
        "endurance_min": t / 60.0,
        "mission_time_min": (mission_complete_time / 60.0) if mission_complete_time else np.nan,
        "energy_Wh": energy_Wh,
        "completed": bool(completed),
        "mean_comp_time": float(np.mean(comp_times)) if comp_times else np.nan,
        "final_soc": soc,
        "fuel_used_Wh": fuel_used_Wh,
        "fuel_remaining_Wh": FC_FUEL_WH - fuel_used_Wh,
        "mean_wind": float(np.mean(wind_speeds)) if wind_speeds else np.nan,
        "max_wind": float(np.max(wind_speeds)) if wind_speeds else np.nan,
        "soc_trace": soc_trace,
        "fuel_trace": fuel_trace,
    }
    return result


In [ ]:
import uav_energy_sim as sim
import importlib
importlib.reload(sim)
print('Simulation module loaded.')


## 2. Monte Carlo experiment driver

In [ ]:
%%writefile run_experiment.py
"""
Monte Carlo driver for the AEMCF simulation.

Usage:
    python3 run_experiment.py --n_runs 50 --out results.csv

Runs baseline_fixed, baseline_offline, aemcf, aemcf_nowind across
n_runs wind seeds (same seeds for every method, so paired tests are
valid). Saves per-run results to CSV plus a SOC/fuel trace for seed 0.
"""
import argparse
import time
import json
import numpy as np
import pandas as pd
from scipy import stats

import uav_energy_sim as sim

METHODS = ["baseline_fixed", "baseline_offline", "aemcf", "aemcf_nowind"]


def run_all(n_runs, max_time=13000.0, seed0=0, verbose=True, out_csv=None, append=False, wind_kwargs=None):
    mpc = sim.PowerMPC(w_balance=15.0, w_fc_band=0.0, w_du=0.02)
    rows = []
    traces = {}
    t_start = time.time()
    header_written = append and out_csv is not None and __import__("os").path.exists(out_csv)
    for i in range(n_runs):
        seed = seed0 + i
        seed_rows = []
        for m in METHODS:
            r = sim.run_mission(m, seed=seed, mpc=mpc, max_time=max_time, wind_kwargs=wind_kwargs)
            if seed == seed0 and not append:
                traces[m] = {"soc": r["soc_trace"], "fuel": r["fuel_trace"]}
            r = {k: v for k, v in r.items() if k not in ("soc_trace", "fuel_trace")}
            seed_rows.append(r)
        rows.extend(seed_rows)
        if out_csv is not None:
            pd.DataFrame(seed_rows).to_csv(out_csv, mode="a" if (append or i > 0) else "w",
                                            header=not header_written, index=False)
            header_written = True
        if verbose:
            elapsed = time.time() - t_start
            print(f"  seed {seed} done ({i+1}/{n_runs}), elapsed {elapsed:.1f}s", flush=True)
    df = pd.DataFrame(rows)
    return df, traces


def summarize(df):
    summary = df.groupby("method").agg(
        endurance_mean=("endurance_min", "mean"),
        endurance_std=("endurance_min", "std"),
        energy_mean=("energy_Wh", "mean"),
        energy_std=("energy_Wh", "std"),
        completion_rate=("completed", "mean"),
        comp_time_mean=("mean_comp_time", "mean"),
    )
    summary["completion_rate"] *= 100.0
    return summary


def paired_tests(df):
    """Paired t-test: AEMCF vs the best baseline, matched by seed."""
    piv_end = df.pivot(index="seed", columns="method", values="endurance_min")
    piv_energy = df.pivot(index="seed", columns="method", values="energy_Wh")
    baselines = ["baseline_fixed", "baseline_offline"]
    best_baseline = piv_end[baselines].mean().idxmax()

    t_end, p_end = stats.ttest_rel(piv_end["aemcf"], piv_end[best_baseline])
    t_en, p_en = stats.ttest_rel(piv_energy["aemcf"], piv_energy[best_baseline])
    t_abl, p_abl = stats.ttest_rel(piv_end["aemcf"], piv_end["aemcf_nowind"])
    return {
        "best_baseline": best_baseline,
        "endurance_vs_best_baseline": {"t": float(t_end), "p": float(p_end)},
        "energy_vs_best_baseline": {"t": float(t_en), "p": float(p_en)},
        "ablation_wind_term_endurance": {"t": float(t_abl), "p": float(p_abl)},
    }


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--n_runs", type=int, default=15)
    ap.add_argument("--max_time", type=float, default=13000.0)
    ap.add_argument("--seed0", type=int, default=0)
    ap.add_argument("--out", type=str, default="results.csv")
    ap.add_argument("--traces_out", type=str, default="traces.json")
    ap.add_argument("--summary_out", type=str, default="summary.csv")
    ap.add_argument("--stats_out", type=str, default="stats.json")
    ap.add_argument("--append", action="store_true")
    args = ap.parse_args()

    df, traces = run_all(args.n_runs, max_time=args.max_time, seed0=args.seed0,
                          out_csv=args.out, append=args.append)
    summary = summarize(df)
    summary.to_csv(args.summary_out)
    tests = paired_tests(df)
    with open(args.stats_out, "w") as f:
        json.dump(tests, f, indent=2)
    with open(args.traces_out, "w") as f:
        json.dump(traces, f)

    print("\n=== SUMMARY ===")
    print(summary)
    print("\n=== SIGNIFICANCE TESTS (paired, n=%d) ===" % args.n_runs)
    print(json.dumps(tests, indent=2))


In [ ]:
import importlib
import run_experiment as rexp
importlib.reload(rexp)
print('Experiment driver loaded.')


## 3. Run the experiment

Each seed runs four controllers: `baseline_fixed` (fixed 50/50 power
split), `baseline_offline` (static equivalent-consumption split,
computed once before flight), `aemcf` (online guidance + QP), and
`aemcf_nowind` (ablation: wind estimate zeroed out). Traces (SOC / fuel
vs. time) are saved for seed 0 only, for the representative-mission
plot.

In [ ]:
N_RUNS = 50
SEED0 = 0
MAX_TIME = 13000.0  # seconds; generous cap so the loiter phase can reach fuel/SOC exhaustion

df, traces = rexp.run_all(N_RUNS, max_time=MAX_TIME, seed0=SEED0, verbose=True,
                           out_csv='results.csv', append=False)
print(df.groupby('method')[['endurance_min', 'energy_Wh']].mean())


## 4. Summary statistics and paired significance tests

In [ ]:
import json
summary = rexp.summarize(df)
print(summary.round(3).to_string())

tests = rexp.paired_tests(df)
print(json.dumps(tests, indent=2))

summary.to_csv('summary.csv')
with open('stats.json', 'w') as f:
    json.dump(tests, f, indent=2)
with open('traces.json', 'w') as f:
    json.dump(traces, f)


## 5. Derived metrics

On-station loiter duration (endurance minus time to finish the search
waypoints) and the fraction of the total onboard energy budget (usable
battery + fuel) consumed before the mission ends.

In [ ]:
df['loiter_min'] = df['endurance_min'] - df['mission_time_min']
df['batt_used_wh'] = (sim.SOC_INIT - df['final_soc']) * sim.BATT_CAP_WH
df['stored_used_wh'] = df['batt_used_wh'] + df['fuel_used_Wh']
TOTAL_BUDGET_WH = sim.BATT_USABLE_WH + sim.FC_FUEL_WH
df['budget_used_pct'] = 100 * df['stored_used_wh'] / TOTAL_BUDGET_WH

print(df.groupby('method')[['loiter_min', 'budget_used_pct']].agg(['mean', 'std']).round(3))


## 6. Plots

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

METHOD_LABEL = {
    "baseline_fixed": "Fixed 50/50 split",
    "baseline_offline": "Static offline split",
    "aemcf": "AEMCF",
    "aemcf_nowind": "AEMCF (wind term removed)",
}
COLORS = {
    "baseline_fixed": "#9e9e9e",
    "baseline_offline": "#4c72b0",
    "aemcf": "#c44e52",
    "aemcf_nowind": "#dd8452",
}
methods_main = ["baseline_fixed", "baseline_offline", "aemcf"]

# Endurance across seeds
fig, ax = plt.subplots(figsize=(6.2, 4.2))
means = [df[df.method == m].endurance_min.mean() for m in methods_main]
stds = [df[df.method == m].endurance_min.std() for m in methods_main]
x = np.arange(len(methods_main))
ax.bar(x, means, yerr=stds, capsize=4, color=[COLORS[m] for m in methods_main], width=0.55)
for i, m in enumerate(methods_main):
    ys = df[df.method == m].endurance_min.values
    xs = np.random.default_rng(0).normal(i, 0.04, size=len(ys))
    ax.scatter(xs, ys, color="black", s=10, zorder=5, alpha=0.6)
ax.set_xticks(x); ax.set_xticklabels([METHOD_LABEL[m] for m in methods_main], rotation=12, ha="right")
ax.set_ylabel("Endurance (minutes)")
ax.set_title(f"Endurance across {N_RUNS} Monte Carlo wind realisations")
fig.tight_layout(); fig.savefig("fig_endurance.png", dpi=200); plt.show()


In [ ]:
# SOC and fuel traces (seed 0)
fig, axes = plt.subplots(2, 1, figsize=(6.6, 5.6), sharex=True)
for m in methods_main:
    soc = np.array(traces[m]["soc"]) * 100.0
    tmin = np.arange(len(soc)) / 60.0
    axes[0].plot(tmin, soc, label=METHOD_LABEL[m], color=COLORS[m], linewidth=1.4)
axes[0].axhline(20, color="k", linestyle=":", linewidth=1, label="SOC floor (20%)")
axes[0].set_ylabel("Battery SOC (%)"); axes[0].legend(fontsize=8, loc="lower left")
axes[0].set_title("Representative mission (seed 0): battery SOC and fuel remaining")
for m in methods_main:
    fuel = np.array(traces[m]["fuel"])
    tmin = np.arange(len(fuel)) / 60.0
    axes[1].plot(tmin, fuel, color=COLORS[m], linewidth=1.4)
axes[1].axhline(0, color="k", linestyle=":", linewidth=1)
axes[1].set_ylabel("Fuel remaining (Wh)"); axes[1].set_xlabel("Time (minutes)")
fig.tight_layout(); fig.savefig("fig_soc_fuel_traces.png", dpi=200); plt.show()


In [ ]:
# Energy consumption vs mean wind speed
fig, ax = plt.subplots(figsize=(6.2, 4.2))
for m in methods_main:
    sub = df[df.method == m]
    ax.scatter(sub.mean_wind, sub.energy_Wh, color=COLORS[m], label=METHOD_LABEL[m], s=28, alpha=0.8)
    if len(sub) > 2:
        z = np.polyfit(sub.mean_wind, sub.energy_Wh, 1)
        xs = np.linspace(sub.mean_wind.min(), sub.mean_wind.max(), 20)
        ax.plot(xs, np.polyval(z, xs), color=COLORS[m], linestyle="--", linewidth=1)
ax.set_xlabel("Mean wind speed over mission (m/s)"); ax.set_ylabel("Total energy consumed (Wh)")
ax.set_title("Energy consumption vs. mean wind speed"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig("fig_energy_vs_wind.png", dpi=200); plt.show()


In [ ]:
# On-station loiter duration vs an example operational requirement
REQ = 80.0
fig, ax = plt.subplots(figsize=(6.2, 4.2))
means_l = [df[df.method == m].loiter_min.mean() for m in methods_main]
stds_l = [df[df.method == m].loiter_min.std() for m in methods_main]
ax.bar(x, means_l, yerr=stds_l, capsize=4, color=[COLORS[m] for m in methods_main], width=0.55)
ax.axhline(REQ, color="k", linestyle=":", linewidth=1.3, label=f"On-station requirement ({REQ:.0f} min)")
ax.set_xticks(x); ax.set_xticklabels([METHOD_LABEL[m] for m in methods_main], rotation=12, ha="right")
ax.set_ylabel("On-station loiter duration (minutes)")
ax.set_title("On-station endurance after completing the search pattern"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig("fig_loiter_vs_requirement.png", dpi=200); plt.show()

for m in methods_main:
    sub = df[df.method == m]
    print(m, "meets requirement in", f"{(sub.loiter_min >= REQ).mean()*100:.0f}%", "of seeds")


## 7. Harsher-wind follow-up experiment

Tests whether the wind-awareness ablation's endurance contribution
(Section 5) grows under stronger, more persistent wind. Uses a harsher
wind process: mean speed 7 m/s (vs. 4 m/s), max gust 15 m/s (vs. 7.5
m/s), and a shorter correlation time of 20 s (vs. 55 s), run for 30
seeds. Takes roughly 15-20 minutes on a Colab CPU runtime.

In [ ]:
HARSH_WIND = dict(mean_speed=7.0, max_speed=15.0, tau=20.0, sigma=2.5)
N_RUNS_HARSH = 30

df_harsh, _ = rexp.run_all(N_RUNS_HARSH, max_time=MAX_TIME, seed0=0, verbose=True,
                            out_csv='harsh_results.csv', append=False,
                            wind_kwargs=HARSH_WIND)
print(df_harsh.groupby('method')[['endurance_min', 'mean_wind', 'max_wind']].mean())


In [ ]:
from scipy import stats as scistats

def paired_diff_stats(df, col='endurance_min'):
    piv = df.pivot(index='seed', columns='method', values=col)
    diff = piv['aemcf'] - piv['aemcf_nowind']
    t, p_t = scistats.ttest_rel(piv['aemcf'], piv['aemcf_nowind'])
    w, p_w = scistats.wilcoxon(piv['aemcf'], piv['aemcf_nowind'])
    return diff.mean(), t, p_t, p_w

mean_mild, t_mild, pt_mild, pw_mild = paired_diff_stats(df)
mean_harsh, t_harsh, pt_harsh, pw_harsh = paired_diff_stats(df_harsh)

print(f"Mild wind   (n={N_RUNS}): mean diff = {mean_mild:+.3f} min, "
      f"t={t_mild:.2f}, p(t-test)={pt_mild:.2e}, p(Wilcoxon)={pw_mild:.2e}")
print(f"Harsher wind (n={N_RUNS_HARSH}): mean diff = {mean_harsh:+.3f} min, "
      f"t={t_harsh:.2f}, p(t-test)={pt_harsh:.2f}, p(Wilcoxon)={pw_harsh:.2f}")


In [ ]:
# Figure 6: wind-awareness ablation effect, mild vs harsher wind
fig, ax = plt.subplots(figsize=(6.0, 4.2))
conditions = [f'Mild wind\n(n={N_RUNS})', f'Harsher wind\n(n={N_RUNS_HARSH})']
means = [mean_mild, mean_harsh]
piv_mild = df.pivot(index='seed', columns='method', values='endurance_min')
piv_harsh = df_harsh.pivot(index='seed', columns='method', values='endurance_min')
sems = [(piv_mild['aemcf']-piv_mild['aemcf_nowind']).std(ddof=1)/np.sqrt(N_RUNS),
        (piv_harsh['aemcf']-piv_harsh['aemcf_nowind']).std(ddof=1)/np.sqrt(N_RUNS_HARSH)]
ax.bar(conditions, means, yerr=sems, capsize=5, color=['#4c72b0', '#c44e52'], width=0.5)
ax.axhline(0, color='k', linewidth=0.8)
ax.set_ylabel('AEMCF minus AEMCF-noWind\nendurance difference (minutes)')
ax.set_title('Wind-awareness contribution vs. wind severity')
fig.tight_layout()
fig.savefig('fig6_wind_ablation_severity.png', dpi=200)
plt.show()


**Result**: contrary to the initial expectation that stronger wind would give wind-aware
guidance more room to help, the wind-awareness contribution did not grow under harsher
wind — it was statistically indistinguishable from zero. This indicates that AEMCF's
principal advantage over the baselines (Section 3-4) comes from the equivalent-consumption
power-balancing objective rather than from wind-aware guidance specifically. See the
accompanying paper's Results and Discussion for the full analysis.

## 8. Extending this experiment

- **More seeds:** increase `N_RUNS` and re-run from Section 3 onward. Runtime scales roughly linearly.
- **Different wind conditions:** edit `WindProcess` in `uav_energy_sim.py` (`mean_speed`, `max_speed`, `tau`).
- **QP weights:** `PowerMPC(w_balance=..., w_fc_band=..., w_du=...)` is instantiated inside `run_experiment.run_all`; edit that function to sweep weights.
- **Vehicle/battery/fuel-cell parameters:** module-level constants near the top of `uav_energy_sim.py`.
- **Mission geometry:** `default_mission()` in `uav_energy_sim.py` defines the waypoint list; `LOITER_RADIUS` sets the holding-pattern radius.
